# Whale Instance Segmentation — YOLOv8 on Thermal Grayscale Images

This notebook documents the full pipeline for training and evaluating a YOLOv8 segmentation model on thermal drone images of whales.

**Pipeline overview:**
1. Dataset preparation — RGB → Grayscale conversion
2. Training (with best model)
3. Final evaluation

---
## 0. Imports & Global Configuration

All paths, constants and thresholds are defined here so they never need to be changed deeper in the notebook.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import yaml
from pathlib import Path

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

# ── Dataset paths ─────────────────────────────────────────────────────────────
DATASET_ROOT_RGB  = "..."            # Original RGB dataset
DATASET_ROOT_GRAY = "./..._grayscale"  # Converted grayscale dataset
DATASET_YAML      = f"{DATASET_ROOT_GRAY}/data.yaml"

# ── Model weights ─────────────────────────────────────────────────────────────
MODEL_TUNED_PATH    = "./runs/segment/train_bestparam/weights/best.pt"     # Final tuned model

# ── Inference thresholds (validated via Experiment 2) ─────────────────────────
CONF_THRESHOLD = 0.25  # Confidence threshold
IOU_THRESHOLD  = 0.30  # IoU threshold — intentionally permissive for partial whale detections

---
## 1. Dataset Preparation

### The RGB → Grayscale issue

The thermal drone images are inherently black-and-white (single-channel), but were saved as 3-channel RGB files by duplicating the same intensity value into R, G and B.

**Why this hurts training:**
- The model wastes capacity learning from redundant colour channels.
- Data augmentation (HSV jitter, colour noise) adds artefacts that don't exist in real thermal imagery, confusing the model.

**Fix:** Convert every image to true 1-channel grayscale before training.

In [ ]:
# Convert all RGB images in the dataset to true 1-channel grayscale.
# Images are overwritten in-place — run this only once on a copy of the dataset.

splits = ["train", "valid", "test"]
total_converted = 0

print("🔄 Starting RGB → Grayscale conversion...")

for split in splits:
    img_dir = Path(DATASET_ROOT_RGB) / split / "images"

    if not img_dir.exists():
        print(f"   ⚠️  Skipped '{split}': directory not found at {img_dir}")
        continue

    files = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.jpeg")) + list(img_dir.glob("*.png"))

    if not files:
        print(f"   ⚠️  Skipped '{split}': no images found.")
        continue

    print(f"   Processing '{split}' ({len(files)} images)...")

    for img_path in files:
        try:
            img = Image.open(img_path)
            if img.mode == "RGB":
                img.convert("L").save(img_path)  # L = 8-bit grayscale
                total_converted += 1
        except Exception as e:
            print(f"   ❌ Error on {img_path.name}: {e}")

print(f"\n✅ Done — {total_converted} images converted to true 1-channel grayscale.")

In [4]:
# Verify dataset split sizes and confirm images are truly grayscale.

print("📂 Dataset split sizes:")
for split in ["train", "valid", "test"]:
    img_dir = Path(DATASET_ROOT_GRAY) / split / "images"
    images = list(img_dir.rglob("*.jpg")) + list(img_dir.rglob("*.png"))
    print(f"   {split:6s}: {len(images)} images")

# Spot-check one image to confirm channel count
sample_img_path = next((Path(DATASET_ROOT_GRAY) / "train" / "images").rglob("*.jpg"))
img = Image.open(sample_img_path)
channels = len(img.getbands())
print(f"\n🔍 Sample image: {sample_img_path.name}")
print(f"   Size: {img.width} x {img.height} px")
print(f"   Channels: {channels} ({'✅ Grayscale' if channels == 1 else '❌ Still RGB — rerun conversion'})")

📂 Dataset split sizes:
   train : 411 images
   valid : 127 images
   test  : 95 images

🔍 Sample image: video_001_event1_A002_500_t-00000s_01_jpg.rf.8a958210d0c7c29fd721f77f48ca9876.jpg
   Size: 432 x 432 px
   Channels: 1 (✅ Grayscale)


---
## 2. Model Training

> **Note:** YOLO automatically resizes images to the nearest multiple of 32.  
> `imgsz=432` → actual training resolution is **448px** (432 rounded up to 448).  
> This is expected behaviour and does not affect results.

In [ ]:
# ── Training with best hyperparameters ──────────────────────────────────
# Load the best hyperparameters found by the tuner and train.

with open("./runs/segment/tune/best_hyperparameters.yaml") as f:
    best_params = yaml.safe_load(f)

print("Best hyperparameters:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

model_tuned = YOLO(MODEL_TUNED_PATH)

model_tuned.train(
    data=DATASET_YAML,
    epochs=150,        
    imgsz=432,
    name="train",
    **best_params      # Inject tuned hyperparameters
)

---
## 3. Evaluation — Model on Test Set

The test set is used **only here**, once, to report the final performance of the tuned model.

In [ ]:
# ── Detailed validation report — Tuned model ─────────────────────────────────
# plots=True generates PR curves and prediction visualisations in runs/segment/val/

final_model = YOLO("./runs/segment/train/weights/best.pt")

metrics = final_model.val(
    data=DATASET_YAML,
    split="val",
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    verbose=True,
    plots=True
)

print("\n" + "=" * 60)
print("📊 VALIDATION RESULTS — Tuned Model (best.pt)")
print("=" * 60)
print(f"mAP50        (Box):  {metrics.box.map50:.4f}")
print(f"mAP50-95     (Box):  {metrics.box.map:.4f}")
print(f"mAP50        (Mask): {metrics.seg.map50:.4f}")
print(f"mAP50-95     (Mask): {metrics.seg.map:.4f}")
print(f"Recall       (Mask): {metrics.seg.r.mean():.4f}")
print(f"Precision    (Mask): {metrics.seg.p.mean():.4f}")
print("=" * 60)